# 01 - Dataset Prep (CPU, no Colab)
This notebook preps fraud data locally (CPU-only), runs EDA artifacts (histograms, boxplots, mean/variance), and sets up baseline models for downstream notebooks.

## 1) Environment Setup
CPU-only workflow; no Colab mounts. Uses pandas/numpy/sklearn/matplotlib. Set paths via env vars when available.

In [ ]:
import os
import math
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional: XGBoost if installed; guarded for CPU-only light params
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

# Paths and config
ROOT = Path(".").resolve()
DATA_PATH = Path(os.getenv("FRAUD_DATA_PATH", ROOT / "data" / "processed" / "transactions.parquet"))
ARTIFACT_DIR = ROOT / "artifacts" / "eda"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RNG_SEED = 42
np.random.seed(RNG_SEED)

print(f"Using data path: {DATA_PATH}")
print(f"Artifacts will be saved to: {ARTIFACT_DIR}")

## 2) Load Dataset (local, CPU-only)
Reads parquet/CSV from DATA_PATH; if missing, synthesizes a small dataset with ~1% fraud ratio to keep runs light.

In [ ]:
def synthesize_transactions(n_rows: int = 50000, fraud_ratio: float = 0.01, rng_seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    labels = rng.choice([0, 1], size=n_rows, p=[1 - fraud_ratio, fraud_ratio])
    amount = rng.gamma(shape=2.0, scale=200.0, size=n_rows)
    oldbalanceOrg = rng.normal(loc=5000, scale=1500, size=n_rows)
    newbalanceOrig = oldbalanceOrg - amount * rng.uniform(0.8, 1.0, size=n_rows)
    oldbalanceDest = rng.normal(loc=2000, scale=1000, size=n_rows)
    newbalanceDest = oldbalanceDest + amount * rng.uniform(0.7, 1.0, size=n_rows)
    tx_type = rng.choice(["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"], size=n_rows)
    return pd.DataFrame({
        "amount": amount,
        "oldbalanceOrg": oldbalanceOrg,
        "newbalanceOrig": newbalanceOrig,
        "oldbalanceDest": oldbalanceDest,
        "newbalanceDest": newbalanceDest,
        "type": tx_type,
        "is_fraud": labels,
    })


def load_dataset(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        print(f"Loaded dataset from {path} with shape {df.shape}")
        return df
    print(f"Data path not found: {path}. Synthesizing sample dataset (~1% fraud).")
    df = synthesize_transactions()
    return df


df_raw = load_dataset(DATA_PATH)
df_raw.head()

## 3) Exploratory Data Analysis
Class balance, summary stats, mean/variance table, histograms, boxplots. Artifacts saved under artifacts/eda.

In [ ]:
# Class balance
class_counts = df_raw['is_fraud'].value_counts(normalize=True)
print("Class balance (fraction):")
print(class_counts)

# Numeric columns for EDA
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != 'is_fraud']

summary = df_raw[feature_cols].agg(['mean', 'var']).T.rename(columns={'var': 'variance'})
feature_stats_path = ARTIFACT_DIR / "feature_stats.csv"
summary.to_csv(feature_stats_path, index=True)
print(f"Saved feature mean/variance to {feature_stats_path}")
summary.head()

In [ ]:
# Histograms for top numerical features
num_for_hist = feature_cols[:8]
fig, axes = plt.subplots(2, math.ceil(len(num_for_hist) / 2), figsize=(16, 8))
axes = axes.flatten()
for ax, col in zip(axes, num_for_hist):
    sns.histplot(df_raw[col], bins=40, kde=False, ax=ax)
    ax.set_title(f"Histogram: {col}")
plt.tight_layout()
hist_path = ARTIFACT_DIR / "dataset_analysis_graphs.png"
fig.savefig(hist_path)
print(f"Saved histograms to {hist_path}")
plt.close(fig)

# Boxplots for outlier-heavy columns (reuse same selection)
fig, axes = plt.subplots(2, math.ceil(len(num_for_hist) / 2), figsize=(16, 8))
axes = axes.flatten()
for ax, col in zip(axes, num_for_hist):
    sns.boxplot(x=df_raw[col], ax=ax, orient="h")
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
box_path = ARTIFACT_DIR / "dataset_boxplots.png"
fig.savefig(box_path)
print(f"Saved boxplots to {box_path}")
plt.close(fig)


## 4) Data Preprocessing
Adds feature engineering (log amount, balance deltas, outlier flags) and builds sklearn preprocessing pipeline.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["balance_delta_org"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balance_delta_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]
    # Outlier flags using IQR
    for col in ["amount", "balance_delta_org", "balance_delta_dest"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"outlier_flag_{col}"] = ((df[col] < lower) | (df[col] > upper)).astype(int)
    return df


df = add_features(df_raw)

cat_cols = ["type"]
num_cols = [c for c in df.columns if c not in cat_cols + ["is_fraud"]]

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

print(f"Numeric cols: {num_cols}")
print(f"Categorical cols: {cat_cols}")

## 5) Train-Test Split
Stratified split to preserve fraud ratio (~1%).

In [ ]:
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train fraud ratio:", y_train.mean().round(4), "Test fraud ratio:", y_test.mean().round(4))

## 6) Model Training
Train lightweight CPU-friendly models: Logistic Regression, Random Forest, optional XGBoost (if installed).

In [ ]:
models = {}

log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=300, class_weight="balanced", n_jobs=-1, C=0.5)),
])
models["log_reg"] = log_reg

rf_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=RNG_SEED,
    )),
])
models["rf"] = rf_clf

if HAS_XGB:
    xgb_clf = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", XGBClassifier(
            max_depth=5,
            n_estimators=200,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            tree_method="hist",
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=RNG_SEED,
        )),
    ])
    models["xgb"] = xgb_clf

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
print("Training complete.")

## 7) Model Evaluation
Compute precision/recall/F1 and plot PR + ROC curves for each model. Save curves to artifacts/eda for reuse.

In [ ]:
eval_rows = []
fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

for name, model in trained_models.items():
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    report = classification_report(y_test, preds, output_dict=True)
    precision, recall, _ = precision_recall_curve(y_test, probs)
    fpr, tpr, _ = roc_curve(y_test, probs)
    pr_auc = auc(recall, precision)
    roc_auc = auc(fpr, tpr)
    eval_rows.append({
        "model": name,
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
    })

    ax_pr.plot(recall, precision, label=f"{name} (AUC={pr_auc:.3f})")
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

ax_pr.set_title("Precision-Recall")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()

ax_roc.set_title("ROC")
ax_roc.set_xlabel("FPR")
ax_roc.set_ylabel("TPR")
ax_roc.legend()

pr_path = ARTIFACT_DIR / "model_pr_curves.png"
roc_path = ARTIFACT_DIR / "model_roc_curves.png"
fig_pr.savefig(pr_path)
fig_roc.savefig(roc_path)
print(f"Saved PR curves to {pr_path}")
print(f"Saved ROC curves to {roc_path}")
plt.close(fig_pr)
plt.close(fig_roc)

pd.DataFrame(eval_rows)

## 8) Inference Demo + Threshold Tuning
Show sample predictions and how threshold shifts precision/recall.

In [ ]:
demo_model_name = "log_reg" if "log_reg" in trained_models else list(trained_models.keys())[0]
demo_model = trained_models[demo_model_name]

probs = demo_model.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0.1, 0.9, 9)
tuning_rows = []
for th in thresholds:
    preds = (probs >= th).astype(int)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    tuning_rows.append({
        "threshold": th,
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
    })

sample_preds = pd.DataFrame({
    "proba": probs[:10],
    "pred@0.5": (probs[:10] >= 0.5).astype(int),
    "label": y_test.iloc[:10].values,
})

tuning_df = pd.DataFrame(tuning_rows)
print(f"Demo model: {demo_model_name}")
print("Sample predictions:")
display(sample_preds)
print("Threshold sweep:")
display(tuning_df)

tuning_path = ARTIFACT_DIR / "threshold_tuning.csv"
tuning_df.to_csv(tuning_path, index=False)
print(f"Saved threshold tuning table to {tuning_path}")

In [ ]:
import subprocess

repo_root = "/workspaces/Fraud-Detection-Agentic-AI"
cmd = ["bash", "scripts/train_local_venv.sh"]

print("Running:", " ".join(cmd), "in", repo_root)
result = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)

print("----- STDOUT -----")
print(result.stdout)
print("----- STDERR -----")
print(result.stderr)
print("EXIT_CODE:", result.returncode)